# DM1 09E — Review-informed model and k selection
# DM1 09E — 复核驱动的模型与 k 选择

This notebook reuses **exactly the modeling population and text representations produced by 09c**. It does not inherit `k=12` as an answer. Instead, TF-IDF/LSA KMeans and weighted MiniLM KMeans are each evaluated independently for every `k=4–20`.

Selection uses the same five random seeds, the same 5,000-row evaluation subset, cluster-size diagnostics, term distinctiveness, and a common review pack. Baseline and advanced models are allowed to select different values of k. The primary model is chosen only after quantitative screening and a comparable interpretation review.

本 notebook 完整复用 09c 的建模样本和文本表示，但不把 `k=12` 当作预设答案。TF-IDF/LSA KMeans 与加权 MiniLM KMeans 分别在 `k=4–20` 上独立评价。两个方法可以选择不同的最佳 k。

In [1]:
from pathlib import Path
import json
import math
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path(r"D:\DM1_project")
INPUT_DIR = ROOT / "v2_clustering_outputs_09c_review_informed"
OUTPUT_DIR = ROOT / "v2_clustering_outputs_09e_model_selection"
CACHE_DIR = INPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASSIGNMENT_PATH = INPUT_DIR / "v2_controlled_cluster_assignments.parquet"
K_VALUES = list(range(4, 21))
SEEDS = [7, 21, 42, 84, 168]
MAIN_SEED = 42
EVALUATION_N = 5_000
COMMENT_WEIGHT = 0.85
CONTEXT_WEIGHT = 0.15

print("09c assignments:", ASSIGNMENT_PATH)
print("09E output:", OUTPUT_DIR)
print("k range:", K_VALUES[0], "to", K_VALUES[-1])
print("seeds:", SEEDS)

09c assignments: D:\DM1_project\v2_clustering_outputs_09c_review_informed\v2_controlled_cluster_assignments.parquet
09E output: D:\DM1_project\v2_clustering_outputs_09e_model_selection
k range: 4 to 20
seeds: [7, 21, 42, 84, 168]


## 1. Reconstruct the exact 09c feature spaces / 重建 09c 的原始特征空间

The fitted 09c vectorizers and LSA transform are reused. The cached weighted MiniLM vectors were created from the same row order using 85% comment and 15% context weights. Assertions prevent silent sample or cache drift.

这里复用 09c 已拟合的向量器、LSA 变换和加权 MiniLM 缓存，并检查样本顺序与规模，避免悄悄改变数据。

In [2]:
df = pd.read_parquet(ASSIGNMENT_PATH)
assert len(df) == 28_230
assert df["comment_id"].is_unique
assert df.groupby("subreddit").size().nunique() == 1

comment_vectorizer = joblib.load(INPUT_DIR / "baseline_comment_tfidf_vectorizer.joblib")
context_vectorizer = joblib.load(INPUT_DIR / "baseline_context_tfidf_vectorizer.joblib")
lsa = joblib.load(INPUT_DIR / "baseline_lsa.joblib")
lsa_normalizer = joblib.load(INPUT_DIR / "baseline_lsa_normalizer.joblib")

X_comment = comment_vectorizer.transform(df["comment_representation_text"])
X_context = context_vectorizer.transform(df["topic_context_text"])
X_tfidf = sparse.hstack([
    X_comment * COMMENT_WEIGHT,
    X_context * CONTEXT_WEIGHT,
], format="csr", dtype=np.float32)
X_tfidf = normalize(X_tfidf, norm="l2", copy=False)
X_lsa = lsa_normalizer.transform(lsa.transform(X_tfidf)).astype(np.float32)

embedding_paths = sorted(CACHE_DIR.glob("weighted_embeddings_*.npy"))
assert len(embedding_paths) == 1, embedding_paths
X_semantic = np.load(embedding_paths[0]).astype(np.float32)
assert X_semantic.shape[0] == len(df)
assert X_semantic.shape[1] == 384

evaluation_indices = np.random.default_rng(MAIN_SEED).choice(
    len(df), size=min(EVALUATION_N, len(df)), replace=False
)

print("Rows:", f"{len(df):,}")
print("LSA shape:", X_lsa.shape)
print("MiniLM shape:", X_semantic.shape)
print("Embedding cache:", embedding_paths[0])

Rows: 28,230
LSA shape: (28230, 100)
MiniLM shape: (28230, 384)
Embedding cache: D:\DM1_project\v2_clustering_outputs_09c_review_informed\cache\weighted_embeddings_7dc459b0cc23146e9aa4.npy


## 2. Evaluate every k with identical controls / 使用完全一致的控制评价所有 k

For each method and k, the notebook records:

- cosine silhouette in both LSA and MiniLM spaces;
- pairwise ARI across five random seeds;
- minimum and maximum cluster shares;
- normalized cluster-size entropy; and
- the mean margin between the assigned and second-closest centroid in the method's native space.

No k is selected in this step. / 此步骤不选择任何 k。

In [3]:
def normalized_centroid_membership(X, labels, centers):
    centers = centers / np.maximum(np.linalg.norm(centers, axis=1, keepdims=True), 1e-12)
    sims = X @ centers.T
    own = sims[np.arange(len(labels)), labels]
    second = np.partition(sims, -2, axis=1)[:, -2]
    return own.astype(np.float32), (own - second).astype(np.float32)


def size_entropy(shares):
    values = shares.to_numpy(dtype=float)
    entropy = -(values * np.log(np.maximum(values, 1e-12))).sum()
    return float(entropy / np.log(len(values)))


spaces = {
    "tfidf_lsa_kmeans": X_lsa,
    "minilm_kmeans": X_semantic,
}
all_models = {}
all_labels = {}
all_membership = {}
metric_rows = []
stability_rows = []

for method, X_native in spaces.items():
    for k in K_VALUES:
        started = time.perf_counter()
        labels_by_seed = {}
        models_by_seed = {}
        for seed in SEEDS:
            model = KMeans(
                n_clusters=k, init="k-means++", n_init=10,
                max_iter=300, random_state=seed,
            )
            labels_by_seed[seed] = model.fit_predict(X_native)
            models_by_seed[seed] = model

        main_model = models_by_seed[MAIN_SEED]
        labels = labels_by_seed[MAIN_SEED]
        shares = pd.Series(labels).value_counts(normalize=True).sort_index()
        membership, margin = normalized_centroid_membership(X_native, labels, main_model.cluster_centers_)

        pair_aris = []
        for i, seed_a in enumerate(SEEDS):
            for seed_b in SEEDS[i + 1:]:
                ari = adjusted_rand_score(labels_by_seed[seed_a], labels_by_seed[seed_b])
                pair_aris.append(ari)
                stability_rows.append({
                    "method": method, "k": k,
                    "seed_a": seed_a, "seed_b": seed_b, "ARI": ari,
                })

        eval_labels = labels[evaluation_indices]
        sil_lsa = silhouette_score(
            X_lsa[evaluation_indices], eval_labels, metric="cosine"
        )
        sil_semantic = silhouette_score(
            X_semantic[evaluation_indices], eval_labels, metric="cosine"
        )
        native_sil = sil_lsa if method == "tfidf_lsa_kmeans" else sil_semantic
        other_sil = sil_semantic if method == "tfidf_lsa_kmeans" else sil_lsa

        metric_rows.append({
            "method": method,
            "k": k,
            "silhouette_lsa": sil_lsa,
            "silhouette_minilm": sil_semantic,
            "silhouette_native": native_sil,
            "silhouette_other_space": other_sil,
            "mean_seed_ari": float(np.mean(pair_aris)),
            "minimum_seed_ari": float(np.min(pair_aris)),
            "minimum_cluster_share": float(shares.min()),
            "maximum_cluster_share": float(shares.max()),
            "normalized_size_entropy": size_entropy(shares),
            "mean_native_membership": float(membership.mean()),
            "mean_native_margin": float(margin.mean()),
            "seconds": time.perf_counter() - started,
        })
        all_models[(method, k)] = main_model
        all_labels[(method, k)] = labels.astype(np.int16)
        all_membership[(method, k)] = membership
        print(
            method, f"k={k}",
            f"native_sil={native_sil:.4f}",
            f"ARI={np.mean(pair_aris):.3f}",
            f"share={shares.min():.3f}-{shares.max():.3f}",
        )

metrics = pd.DataFrame(metric_rows)
stability_pairs = pd.DataFrame(stability_rows)
metrics

D:\python 3.13.7\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系统找不到指定的文件。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "D:\python 3.13.7\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "D:\python 3.13.7\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "D:\python 3.13.7\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "D:\python 3.13.7\Lib\subprocess.py", line 1039

tfidf_lsa_kmeans k=4 native_sil=0.0645 ARI=0.823 share=0.080-0.395


tfidf_lsa_kmeans k=5 native_sil=0.0715 ARI=0.997 share=0.077-0.341


tfidf_lsa_kmeans k=6 native_sil=0.0775 ARI=0.700 share=0.065-0.303


tfidf_lsa_kmeans k=7 native_sil=0.0807 ARI=0.689 share=0.051-0.315


...

minilm_kmeans k=17 native_sil=0.0475 ARI=0.910 share=0.027-0.087


minilm_kmeans k=18 native_sil=0.0498 ARI=0.769 share=0.028-0.089


minilm_kmeans k=19 native_sil=0.0480 ARI=0.795 share=0.026-0.087


minilm_kmeans k=20 native_sil=0.0481 ARI=0.903 share=0.022-0.087


,method,k,silhouette_lsa,silhouette_minilm,silhouette_native,silhouette_other_space,mean_seed_ari,minimum_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,mean_native_membership,mean_native_margin,seconds
0,tfidf_lsa_kmeans,4,0.064490,-0.017222,0.064490,-0.017222,0.822840,0.665833,0.079844,0.395289,0.902001,0.415158,0.139180,2.620414
1,tfidf_lsa_kmeans,5,0.071463,-0.009210,0.071463,-0.009210,0.996868,0.993304,0.076727,0.341410,0.942852,0.430127,0.137717,2.656428
2,tfidf_lsa_kmeans,6,0.077550,-0.006124,0.077550,-0.006124,0.700228,0.568957,0.065143,0.302976,0.932166,0.442741,0.143614,2.882557
3,tfidf_lsa_kmeans,7,0.080718,-0.023873,0.080718,-0.023873,0.689022,0.546585,0.050549,0.314984,0.908440,0.453311,0.149928,2.983531
4,tfidf_lsa_kmeans,8,0.086603,0.010256,0.086603,0.010256,0.680961,0.507961,0.060468,0.252674,0.932269,0.464343,0.153294,3.238454
5,tfidf_lsa_kmeans,9,0.088354,-0.003308,0.088354,-0.003308,0.653425,0.515486,0.042650,0.196635,0.944930,0.474237,0.151766,3.496272
6,tfidf_lsa_kmeans,10,0.095652,-0.004437,0.095652,-0.004437,0.672667,0.547068,0.044810,0.253170,0.933877,0.483194,0.157733,3.950716
7,tfidf_lsa_kmeans,11,0.098418,0.000945,0.098418,0.000945,0.801473,0.654599,0.041162,0.202869,0.949324,0.491257,0.156662,4.413212
8,tfidf_lsa_kmeans,12,0.100916,0.001804,0.100916,0.001804,0.816102,0.727123,0.030677,0.202196,0.943078,0.497437,0.159518,4.973226
9,tfidf_lsa_kmeans,13,0.104403,0.000142,0.104403,0.000142,0.846342,0.767841,0.031243,0.180198,0.948394,0.504902,0.159213,5.371828


## 3. Add representation-independent topic-term diagnostics / 添加跨表示的主题词诊断

All assignments are described from the same cleaned comment text. Class-based TF-IDF identifies the top terms for every cluster. Two diagnostics are added:

- **topic diversity**: the share of unique terms among all top-10 topic terms;
- **term exclusivity**: how strongly a selected term belongs to one cluster rather than appearing throughout the corpus.

These are interpretation diagnostics, not substitutes for reading comments. / 它们是解释性诊断，不能替代评论阅读。

In [4]:
domain_stop = {
    "climate", "change", "global", "people", "person", "just", "like", "don",
    "doesn", "didn", "isn", "aren", "think", "really", "thing", "things",
    "way", "know", "make", "makes", "want", "need", "time", "times", "year",
    "years", "world", "say", "says", "said", "ve", "re", "ll", "im", "reddit",
    "https", "http", "com", "amp", "would", "could", "also", "much", "many",
    "get", "got", "going", "one", "even",
}
term_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=sorted(set(ENGLISH_STOP_WORDS).union(domain_stop)),
    ngram_range=(1, 2), min_df=10, max_df=0.85,
    max_features=40_000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9\-]{1,}\b",
)
X_terms = term_vectorizer.fit_transform(df["topic_comment_text"].fillna(""))
term_names = np.asarray(term_vectorizer.get_feature_names_out())
global_counts = np.asarray(X_terms.sum(axis=0)).ravel()


def topic_term_diagnostics(labels, k, top_n=10):
    membership_matrix = sparse.csr_matrix(
        (np.ones(len(labels)), (labels, np.arange(len(labels)))),
        shape=(k, len(labels)),
    )
    cluster_counts = (membership_matrix @ X_terms).tocsr()
    class_lengths = np.asarray(cluster_counts.sum(axis=1)).ravel()
    idf = np.log1p(class_lengths.mean() / np.maximum(global_counts, 1))
    rows = []
    chosen_all = []
    exclusivities = []
    for cluster in range(k):
        counts = cluster_counts.getrow(cluster).toarray().ravel()
        scores = (counts / max(class_lengths[cluster], 1)) * idf
        order = np.argsort(scores)[::-1]
        chosen = []
        for idx in order:
            term = term_names[idx]
            if any(term == old or term in old or old in term for old in chosen):
                continue
            chosen.append(term)
            exclusivity = float(counts[idx] / max(global_counts[idx], 1))
            exclusivities.append(exclusivity)
            rows.append({
                "cluster": cluster,
                "rank": len(chosen),
                "term": term,
                "ctfidf_score": float(scores[idx]),
                "term_exclusivity": exclusivity,
            })
            if len(chosen) == top_n:
                break
        chosen_all.extend(chosen)
    diversity = len(set(chosen_all)) / max(len(chosen_all), 1)
    return pd.DataFrame(rows), float(diversity), float(np.mean(exclusivities))


term_frames = []
term_metric_rows = []
for (method, k), labels in all_labels.items():
    term_df, diversity, exclusivity = topic_term_diagnostics(labels, k)
    term_df.insert(0, "k", k)
    term_df.insert(0, "method", method)
    term_frames.append(term_df)
    term_metric_rows.append({
        "method": method, "k": k,
        "topic_diversity": diversity,
        "mean_term_exclusivity": exclusivity,
    })

all_topic_terms = pd.concat(term_frames, ignore_index=True)
metrics = metrics.merge(pd.DataFrame(term_metric_rows), on=["method", "k"], validate="one_to_one")
metrics

,method,k,silhouette_lsa,silhouette_minilm,silhouette_native,silhouette_other_space,mean_seed_ari,minimum_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,mean_native_membership,mean_native_margin,seconds,topic_diversity,mean_term_exclusivity
0,tfidf_lsa_kmeans,4,0.064490,-0.017222,0.064490,-0.017222,0.822840,0.665833,0.079844,0.395289,0.902001,0.415158,0.139180,2.620414,0.875000,0.497032
1,tfidf_lsa_kmeans,5,0.071463,-0.009210,0.071463,-0.009210,0.996868,0.993304,0.076727,0.341410,0.942852,0.430127,0.137717,2.656428,0.880000,0.480683
2,tfidf_lsa_kmeans,6,0.077550,-0.006124,0.077550,-0.006124,0.700228,0.568957,0.065143,0.302976,0.932166,0.442741,0.143614,2.882557,0.850000,0.424811
3,tfidf_lsa_kmeans,7,0.080718,-0.023873,0.080718,-0.023873,0.689022,0.546585,0.050549,0.314984,0.908440,0.453311,0.149928,2.983531,0.828571,0.371654
4,tfidf_lsa_kmeans,8,0.086603,0.010256,0.086603,0.010256,0.680961,0.507961,0.060468,0.252674,0.932269,0.464343,0.153294,3.238454,0.787500,0.389952
5,tfidf_lsa_kmeans,9,0.088354,-0.003308,0.088354,-0.003308,0.653425,0.515486,0.042650,0.196635,0.944930,0.474237,0.151766,3.496272,0.788889,0.359363
6,tfidf_lsa_kmeans,10,0.095652,-0.004437,0.095652,-0.004437,0.672667,0.547068,0.044810,0.253170,0.933877,0.483194,0.157733,3.950716,0.730000,0.333950
7,tfidf_lsa_kmeans,11,0.098418,0.000945,0.098418,0.000945,0.801473,0.654599,0.041162,0.202869,0.949324,0.491257,0.156662,4.413212,0.736364,0.324427
8,tfidf_lsa_kmeans,12,0.100916,0.001804,0.100916,0.001804,0.816102,0.727123,0.030677,0.202196,0.943078,0.497437,0.159518,4.973226,0.733333,0.317757
9,tfidf_lsa_kmeans,13,0.104403,0.000142,0.104403,0.000142,0.846342,0.767841,0.031243,0.180198,0.948394,0.504902,0.159213,5.371828,0.700000,0.302898


## 4. Transparent screening and shortlisting / 透明筛选与候选名单

Before ranking, solutions are required to have no cluster smaller than 2%, no cluster larger than 40%, and mean seed ARI of at least 0.70. Within each method, seven diagnostics are min-max scaled and combined with declared weights. This score creates a shortlist; it is not treated as an automatic substantive verdict.

在排序前，方案必须满足：最小簇不低于 2%、最大簇不高于 40%、平均种子 ARI 不低于 0.70。综合分只用于形成候选名单，不能自动替代语义复核。

In [5]:
SELECTION_WEIGHTS = {
    "mean_seed_ari": 0.25,
    "silhouette_native": 0.20,
    "mean_term_exclusivity": 0.15,
    "topic_diversity": 0.10,
    "normalized_size_entropy": 0.10,
    "minimum_cluster_share": 0.10,
    "mean_native_margin": 0.10,
}

metrics["passes_guardrails"] = (
    metrics["minimum_cluster_share"].ge(0.02)
    & metrics["maximum_cluster_share"].le(0.40)
    & metrics["mean_seed_ari"].ge(0.70)
)


def minmax(series):
    lo, hi = series.min(), series.max()
    if math.isclose(float(lo), float(hi)):
        return pd.Series(1.0, index=series.index)
    return (series - lo) / (hi - lo)


scored_parts = []
for method, group in metrics.groupby("method", sort=False):
    group = group.copy()
    eligible = group["passes_guardrails"]
    for column in SELECTION_WEIGHTS:
        group[column + "_scaled"] = minmax(group[column])
    group["quantitative_score"] = sum(
        weight * group[column + "_scaled"]
        for column, weight in SELECTION_WEIGHTS.items()
    )
    group["quantitative_rank"] = group["quantitative_score"].rank(
        method="min", ascending=False
    ).astype(int)
    scored_parts.append(group)

selection_scores = pd.concat(scored_parts, ignore_index=True)
shortlist = (
    selection_scores.loc[selection_scores["passes_guardrails"]]
    .sort_values(["method", "quantitative_score"], ascending=[True, False])
    .groupby("method", as_index=False, group_keys=False)
    .head(3)
    .copy()
)

display_columns = [
    "method", "k", "passes_guardrails", "quantitative_rank", "quantitative_score",
    "silhouette_lsa", "silhouette_minilm", "mean_seed_ari",
    "minimum_cluster_share", "maximum_cluster_share", "normalized_size_entropy",
    "topic_diversity", "mean_term_exclusivity", "mean_native_margin",
]
display(selection_scores[display_columns].sort_values(["method", "quantitative_rank"]))
print("Shortlist / 候选名单")
display(shortlist[display_columns])

,method,k,passes_guardrails,quantitative_rank,quantitative_score,silhouette_lsa,silhouette_minilm,mean_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,topic_diversity,mean_term_exclusivity,mean_native_margin
19,minilm_kmeans,6,True,1,0.786905,0.038471,0.068002,0.994809,0.067552,0.236132,0.969295,0.883333,0.474746,0.083036
20,minilm_kmeans,7,True,2,0.746072,0.034356,0.064836,0.987070,0.066844,0.188169,0.982086,0.857143,0.440551,0.082417
17,minilm_kmeans,4,True,3,0.741580,0.041604,0.082186,0.828095,0.070067,0.354410,0.909442,0.900000,0.590811,0.094045
22,minilm_kmeans,9,True,4,0.669833,0.032322,0.070545,0.975249,0.048176,0.155898,0.974821,0.788889,0.396752,0.082854
21,minilm_kmeans,8,True,5,0.654351,0.036934,0.066794,0.973893,0.047680,0.177471,0.969404,0.800000,0.416407,0.083149
18,minilm_kmeans,5,True,6,0.608766,0.039640,0.077196,0.731102,0.068650,0.306518,0.948402,0.860000,0.513169,0.092761
24,minilm_kmeans,11,True,7,0.525736,0.032292,0.056533,0.995877,0.040914,0.134998,0.975983,0.727273,0.339634,0.078774
25,minilm_kmeans,12,True,8,0.503489,0.030650,0.057099,0.973460,0.040666,0.126461,0.979363,0.733333,0.334578,0.076385
26,minilm_kmeans,13,True,9,0.413114,0.030238,0.057717,0.906057,0.029508,0.125292,0.972887,0.730769,0.331515,0.077477
23,minilm_kmeans,10,True,10,0.394830,0.031135,0.057029,0.813163,0.047821,0.137124,0.981298,0.760000,0.360261,0.076824


Shortlist / 候选名单


,method,k,passes_guardrails,quantitative_rank,quantitative_score,silhouette_lsa,silhouette_minilm,mean_seed_ari,minimum_cluster_share,maximum_cluster_share,normalized_size_entropy,topic_diversity,mean_term_exclusivity,mean_native_margin
19,minilm_kmeans,6,True,1,0.786905,0.038471,0.068002,0.994809,0.067552,0.236132,0.969295,0.883333,0.474746,0.083036
20,minilm_kmeans,7,True,2,0.746072,0.034356,0.064836,0.987070,0.066844,0.188169,0.982086,0.857143,0.440551,0.082417
17,minilm_kmeans,4,True,3,0.741580,0.041604,0.082186,0.828095,0.070067,0.354410,0.909442,0.900000,0.590811,0.094045
1,tfidf_lsa_kmeans,5,True,1,0.686655,0.071463,-0.009210,0.996868,0.076727,0.341410,0.942852,0.880000,0.480683,0.137717
9,tfidf_lsa_kmeans,13,True,2,0.534263,0.104403,0.000142,0.846342,0.031243,0.180198,0.948394,0.700000,0.302898,0.159213
7,tfidf_lsa_kmeans,11,True,3,0.514802,0.098418,0.000945,0.801473,0.041162,0.202869,0.949324,0.736364,0.324427,0.156662


## 5. Create one comparable interpretation pack / 生成统一解释性复核包

For every shortlisted solution, each cluster contributes three centroid-nearest comments and three random comments from distinct posts. The same sampling rule is used for both methods and every k. This pack supports a blinded comparison of clarity, overlap, and residual structure.

每个候选方案的每个簇都抽取 3 条中心评论和 3 条不同帖子的随机评论，所有方法和 k 使用同一规则。

In [6]:
review_rows = []
profile_rows = []

for candidate in shortlist.itertuples(index=False):
    method, k = candidate.method, int(candidate.k)
    labels = all_labels[(method, k)]
    strength = all_membership[(method, k)]
    terms = all_topic_terms.loc[
        all_topic_terms["method"].eq(method) & all_topic_terms["k"].eq(k)
    ]
    term_map = terms.groupby("cluster")["term"].apply(lambda x: ", ".join(x)).to_dict()

    candidate_df = df[[
        "comment_id", "post_id", "subreddit", "post_title", "clean_text",
        "comment_representation_text", "topic_context_text", "review_gate_score",
    ]].copy()
    candidate_df["cluster"] = labels
    candidate_df["membership_strength"] = strength

    for cluster, group in candidate_df.groupby("cluster"):
        unique_posts = group.sort_values("membership_strength", ascending=False).drop_duplicates("post_id")
        centroid = unique_posts.head(3)
        remaining = unique_posts.drop(index=centroid.index)
        random = remaining.sample(n=min(3, len(remaining)), random_state=MAIN_SEED + int(cluster))
        profile_rows.append({
            "method": method, "k": k, "cluster": int(cluster),
            "size": int(len(group)),
            "share": float(len(group) / len(candidate_df)),
            "mean_membership": float(group["membership_strength"].mean()),
            "top_terms": term_map[int(cluster)],
        })
        for sample_type, chosen in [("centroid", centroid), ("random", random)]:
            for rank, row in enumerate(chosen.itertuples(index=False), start=1):
                review_rows.append({
                    "method": method, "k": k, "cluster": int(cluster),
                    "sample_type": sample_type, "rank": rank,
                    "comment_id": row.comment_id, "post_id": row.post_id,
                    "subreddit": row.subreddit, "post_title": row.post_title,
                    "clean_text": row.clean_text,
                    "comment_representation_text": row.comment_representation_text,
                    "topic_context_text": row.topic_context_text,
                    "review_gate_score": row.review_gate_score,
                    "membership_strength": row.membership_strength,
                    "top_terms": term_map[int(cluster)],
                })

candidate_profiles = pd.DataFrame(profile_rows)
candidate_review_pack = pd.DataFrame(review_rows)
display(candidate_profiles.head(30))
print("Review rows:", len(candidate_review_pack))

,method,k,cluster,size,share,mean_membership,top_terms
0,minilm_kmeans,6,0,5729,0.202940,0.626398,"solar, energy, power, nuclear, wind, grid, ren..."
1,minilm_kmeans,6,1,4809,0.170351,0.534302,"oil, energy, trump, fossil, gas, companies, mo..."
2,minilm_kmeans,6,2,6666,0.236132,0.650406,"warming, planet, problem, humans, real, life, ..."
3,minilm_kmeans,6,3,1907,0.067552,0.637232,"ev, car, electric, battery, vehicles, ice, bat..."
4,minilm_kmeans,6,4,4393,0.155615,0.583816,"warming, temperature, ice, weather, earth, dat..."
5,minilm_kmeans,6,5,4726,0.167411,0.616999,"carbon, co2, emissions, fossil, energy, use, a..."
6,minilm_kmeans,7,0,4075,0.144350,0.577011,"oil, energy, fossil, gas, trump, companies, fu..."
7,minilm_kmeans,7,1,5312,0.188169,0.626072,"solar, energy, power, nuclear, wind, grid, ele..."
8,minilm_kmeans,7,2,4007,0.141941,0.630763,"co2, carbon, emissions, atmosphere, fossil, me..."
9,minilm_kmeans,7,3,4051,0.143500,0.568469,"planet, life, human, food, population, water, ..."


Review rows: 276


## 6. Provisional quantitative winners and exports / 暂定量化优胜方案与导出

The highest quantitative score within each method is exported as a **provisional winner**. The final 09E decision must also cite the comparable interpretation pack. This separation prevents a weighted score from being misrepresented as human interpretability.

每种方法的最高量化分先作为“暂定优胜方案”导出。09E 的最终决定还必须引用统一解释性复核包，避免把加权分数冒充人工可解释性。

In [7]:
provisional_winners = (
    selection_scores.loc[selection_scores["passes_guardrails"]]
    .sort_values(["method", "quantitative_score"], ascending=[True, False])
    .groupby("method", as_index=False, group_keys=False)
    .head(1)
    .copy()
)

metrics.to_csv(OUTPUT_DIR / "all_k_raw_metrics.csv", index=False, encoding="utf-8-sig")
stability_pairs.to_csv(OUTPUT_DIR / "all_k_stability_pairs.csv", index=False, encoding="utf-8-sig")
selection_scores.to_csv(OUTPUT_DIR / "all_k_selection_scores.csv", index=False, encoding="utf-8-sig")
shortlist.to_csv(OUTPUT_DIR / "shortlisted_solutions.csv", index=False, encoding="utf-8-sig")
all_topic_terms.to_csv(OUTPUT_DIR / "all_k_topic_terms.csv", index=False, encoding="utf-8-sig")
candidate_profiles.to_csv(OUTPUT_DIR / "candidate_topic_profiles.csv", index=False, encoding="utf-8-sig")
candidate_review_pack.to_csv(OUTPUT_DIR / "candidate_interpretation_review_pack.csv", index=False, encoding="utf-8-sig")

np.savez_compressed(
    OUTPUT_DIR / "all_k_labels.npz",
    **{f"{method}__k{k}": labels for (method, k), labels in all_labels.items()},
)

for row in provisional_winners.itertuples(index=False):
    joblib.dump(
        all_models[(row.method, int(row.k))],
        OUTPUT_DIR / f"provisional_{row.method}_k{int(row.k)}.joblib",
    )

decision = {
    "status": "provisional_pending_comparable_interpretation_review",
    "data_source": str(ASSIGNMENT_PATH),
    "n_comments": int(len(df)),
    "k_range": [min(K_VALUES), max(K_VALUES)],
    "seeds": SEEDS,
    "guardrails": {
        "minimum_cluster_share": 0.02,
        "maximum_cluster_share": 0.40,
        "minimum_mean_seed_ari": 0.70,
    },
    "selection_weights": SELECTION_WEIGHTS,
    "provisional_winners": provisional_winners[
        ["method", "k", "quantitative_score", "quantitative_rank"]
    ].to_dict("records"),
    "shortlist": shortlist[["method", "k", "quantitative_score"]].to_dict("records"),
    "interpretation_review_required": True,
    "note": "No k is inherited from 09. Final selection requires equal-rule review of shortlisted solutions.",
}
(OUTPUT_DIR / "provisional_selection_decision.json").write_text(
    json.dumps(decision, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(json.dumps(decision, ensure_ascii=False, indent=2))
print("Saved to:", OUTPUT_DIR)

{
  "status": "provisional_pending_comparable_interpretation_review",
  "data_source": "D:\\DM1_project\\v2_clustering_outputs_09c_review_informed\\v2_controlled_cluster_assignments.parquet",
  "n_comments": 28230,
  "k_range": [
    4,
    20
  ],
  "seeds": [
    7,
    21,
    42,
    84,
    168
  ],
  "guardrails": {
    "minimum_cluster_share": 0.02,
    "maximum_cluster_share": 0.4,
    "minimum_mean_seed_ari": 0.7
  },
  "selection_weights": {
    "mean_seed_ari": 0.25,
    "silhouette_native": 0.2,
    "mean_term_exclusivity": 0.15,
    "topic_diversity": 0.1,
    "normalized_size_entropy": 0.1,
    "minimum_cluster_share": 0.1,
    "mean_native_margin": 0.1
  },
  "provisional_winners": [
    {
      "method": "minilm_kmeans",
      "k": 6,
      "quantitative_score": 0.786904665091311,
      "quantitative_rank": 1
    },
    {
      "method": "tfidf_lsa_kmeans",
      "k": 5,
      "quantitative_score": 0.6866545618132642,
      "quantitative_rank": 1
    }
  ],
  "shortlist

## Interpretation rule / 解释规则

Do not call the provisional winner the final model until the shortlisted review pack has been examined with the same rubric. The final decision should record, for every shortlisted solution, the number of clear, broad, mixed, duplicate, and residual clusters and the fit of random comments. 09F must load the final—not provisional—decision.

在使用同一量表完成候选方案复核前，不得把暂定优胜方案称为最终模型。最终决定必须记录每个候选方案的清晰、宽泛、混合、重复和残余簇数量，以及随机评论的匹配情况。09F 只能读取最终决定。

## 7. Final decision after comparable interpretation review / 统一解释性复核后的最终决定

The score-only shortlist was supplemented with MiniLM k=9 because it was the first outside the top three, passed every guardrail, and added substantively distinct rather than fragmentary topics. Equal-rule centroid and random samples were reviewed for clarity, breadth, mixing, duplication, and residual structure.

量化前三之外补充复核了 MiniLM k=9：它通过全部门槛，并增加了实质性主题而非碎片。最终 advanced 选择 k=9，baseline 选择 k=11。

### Reproduce the final export / 复现最终导出

Run `outputs/finalize_09e_selection.py` after the comparable interpretation review. The saved final decision is:

- baseline: `tfidf_lsa_kmeans`, `k=11`;
- advanced and primary model: `minilm_kmeans`, `k=9`;
- status: `final_after_comparable_interpretation_review`.

The full auditable decision is saved in `v2_clustering_outputs_09e_model_selection/final_selection_decision.json`.